# Private AI assistant builder (Kaggle T4)
Files in → parse → clean → chunk → index (RAG) → optional QLoRA → test → Gradio chat → wipe.

**Before you start**
- Accelerator: GPU T4 x2 (only one GPU is used). Internet: ON.
- Put the client's files in a **private** Kaggle Dataset and set `DATA_DIR` below.
- Run this interactively. **Do not use "Save Version / Commit"**: it stores logs and outputs.
- Download the deliverables, then run the last cell and delete the Kaggle Dataset.

In [ ]:
# Config
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # must be set before torch loads
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ORG        = "Client Name"                        # shown in the chat title
CLIENT     = "client1"                            # used in output file names
DATA_DIR   = "/kaggle/input/client-data"          # folder with the client's files
MODEL_SIZE = "medium"                             # small | medium | large  (matches the website form)
MODE       = "rag_plus_ft"                        # rag_only (fast) | rag_plus_ft (adds QLoRA)
SESSION_MINUTES = 20                              # how long the public chat stays up

TEST_QUESTIONS = [                                # write 5-10 real questions the client would ask
    "What are your opening hours?",
    "What is the refund policy?",
]

# Advanced
MODELS = {
    "small":  "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "medium": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    "large":  "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   # slow and tight on a T4
}
EMBED_MODEL   = "BAAI/bge-m3"                     # multilingual (Hindi + English)
MAX_CHARS     = 1000                              # chunk size in characters
MIN_CHARS     = 80                                # drop tiny fragments
TOP_K         = 4                                 # chunks given to the model per question
MAX_SEQ       = 2048
QA_CHUNK_LIMIT = 250                              # max chunks used to generate training Q&A
QA_PER_CHUNK  = 2
NEG_RATIO     = 0.10                              # share of "not in the documents" examples
MIN_EXAMPLES  = 60                                # below this, skip fine-tuning (too little data)
EPOCHS        = 2
SAVE_ADAPTER  = True
EXPORT_GGUF   = False                             # True = also build a Q4_K_M GGUF (adds ~10 min)
SCRUB = {"aadhaar": True, "pan": True, "card": True, "phone": False, "email": False}

In [ ]:
# Install
get_ipython().run_line_magic(
    "pip",
    "install -q unsloth 'markitdown[pdf,docx,pptx,xlsx,xls]' rapidocr-onnxruntime pymupdf "
    "faiss-cpu rank_bm25 datasketch sentence-transformers gradio"
)

In [ ]:
# Imports (unsloth first, on purpose)
from unsloth import FastLanguageModel
import re, json, time, random, secrets, shutil, glob, gc
import numpy as np, torch, faiss
from datasets import Dataset
T0 = time.time()
random.seed(42)

In [ ]:
# 1. Parse every file type into text
from markitdown import MarkItDown
from rapidocr_onnxruntime import RapidOCR
import fitz  # pymupdf

mdconv, ocr_engine = MarkItDown(), RapidOCR()
IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}

def ocr(img):
    res, _ = ocr_engine(img)
    return "\n".join(r[1] for r in res) if res else ""

def ocr_pdf(path):
    pages = []
    for page in fitz.open(path):
        pix = page.get_pixmap(dpi=150)
        arr = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
        pages.append(ocr(arr))
    return "\n\n".join(pages)

def parse_file(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in {".txt", ".md"}:
        return open(path, encoding="utf-8", errors="ignore").read()
    if ext in IMG_EXT:
        return ocr(path)
    text = mdconv.convert(path).text_content
    if ext == ".pdf" and len(text.strip()) < 200:      # looks scanned
        text = ocr_pdf(path)
    return text

docs, failed = {}, []
for p in sorted(glob.glob(f"{DATA_DIR}/**/*", recursive=True)):
    if not os.path.isfile(p):
        continue
    name = os.path.relpath(p, DATA_DIR)
    try:
        t = parse_file(p)
        if t and t.strip():
            docs[name] = t
        else:
            failed.append(name)
    except Exception as e:
        failed.append(f"{name} ({type(e).__name__})")
print(f"Parsed {len(docs)} files. Failed or empty: {len(failed)}")
for f_ in failed:
    print("  -", f_)

In [ ]:
# 2. Clean, scrub private numbers, chunk, remove duplicates
PII = {
    "card":    (r"\b(?:\d[ -]?){13,16}\b", "[CARD]"),
    "aadhaar": (r"\b\d{4}\s?\d{4}\s?\d{4}\b", "[AADHAAR]"),
    "pan":     (r"\b[A-Z]{5}\d{4}[A-Z]\b", "[PAN]"),
    "phone":   (r"(?:\+91[\s-]?)?\b[6-9]\d{9}\b", "[PHONE]"),
    "email":   (r"[\w.+-]+@[\w-]+\.[\w.-]+", "[EMAIL]"),
}
def clean(t):
    t = t.replace("\r", "")
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    for key, on in SCRUB.items():
        if on:
            t = re.sub(PII[key][0], PII[key][1], t)
    return t.strip()

def chunk_doc(text, src):
    """Pack lines into chunks; repeat the heading and table header in each chunk."""
    out, buf, heading, thead, in_tab = [], [], "", [], False
    def seed():
        return ([heading] if heading else []) + (thead if in_tab else [])
    def flush():
        nonlocal buf
        body = "\n".join(buf).strip()
        if len(body) >= MIN_CHARS:
            out.append({"src": src, "text": body})
        buf = seed()
    for line in text.split("\n"):
        if line.startswith("#"):
            flush()
            heading, thead, in_tab = line, [], False
            buf = [line]
            continue
        if line.startswith("|"):
            if not in_tab:
                in_tab, thead = True, [line]
            elif len(thead) < 2:
                thead.append(line)
        elif line.strip():
            in_tab, thead = False, []
        if sum(map(len, buf)) + len(line) > MAX_CHARS and len(buf) > len(seed()):
            flush()
        buf.append(line)
    flush()
    return out

chunks = []
for name, t in docs.items():
    chunks += chunk_doc(clean(t), name)

from datasketch import MinHash, MinHashLSH
lsh, kept = MinHashLSH(threshold=0.9, num_perm=64), []
for i, c in enumerate(chunks):
    w = re.findall(r"\w+", c["text"].lower())
    m = MinHash(num_perm=64)
    for s in {" ".join(w[j:j + 4]) for j in range(max(1, len(w) - 3))}:
        m.update(s.encode())
    if lsh.query(m):
        continue
    lsh.insert(str(i), m)
    kept.append(c)
print(f"Chunks: {len(chunks)} -> {len(kept)} after duplicate removal")
chunks = kept

In [ ]:
# 3. Build the search index (dense + keyword, merged)
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

emb = SentenceTransformer(EMBED_MODEL, device="cuda")
emb.max_seq_length = 512
emb.half()
vecs = emb.encode([c["text"] for c in chunks], batch_size=32,
                  normalize_embeddings=True, show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(vecs.shape[1])
index.add(vecs)
words = lambda t: re.findall(r"\w+", t.lower())
bm25 = BM25Okapi([words(c["text"]) for c in chunks])

def retrieve(q, k=TOP_K):
    qv = emb.encode([q], normalize_embeddings=True).astype("float32")
    _, di = index.search(qv, min(20, len(chunks)))
    bi = np.argsort(-bm25.get_scores(words(q)))[:20]
    score = {}
    for ids in (di[0], bi):
        for r, i in enumerate(ids):
            if i >= 0:
                score[int(i)] = score.get(int(i), 0) + 1 / (60 + r)
    return [chunks[i] for i in sorted(score, key=score.get, reverse=True)[:k]]

print("Index ready:", len(chunks), "chunks")

In [ ]:
# 4. Load the language model and define prompt helpers
MODEL_NAME = MODELS[MODEL_SIZE]
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ, dtype=torch.float16, load_in_4bit=True)
tokenizer.padding_side = "left"

SYSTEM = (f"You are the assistant for {ORG}. Answer only from the context given. "
          "If the answer is not in the context, say you don't have that information. "
          "Be short and clear, and end with 'Source: <file name>'.")
NO_INFO = "I don't have that information in the documents."

def build_messages(q, ctx):
    c = "\n\n".join(f"[{x['src']}]\n{x['text']}" for x in ctx)
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n{c}\n\nQuestion: {q}"}]

def generate(msgs_list, max_new=300, bs=4):
    outs = []
    for i in range(0, len(msgs_list), bs):
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                 for m in msgs_list[i:i + bs]]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            o = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        outs += tokenizer.batch_decode(o[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [x.strip() for x in outs]

In [ ]:
# 5. Make training data from the documents (only in rag_plus_ft mode)
examples = []
if MODE == "rag_plus_ft":
    sample = random.sample(chunks, min(QA_CHUNK_LIMIT, len(chunks)))
    prompts = [[{"role": "user", "content":
        f"Read the text and write {QA_PER_CHUNK} question-and-answer pairs that a customer or staff "
        "member might really ask. Use only facts in the text. Reply with only JSON like "
        '[{"q": "...", "a": "..."}].\n\nText:\n' + c["text"]}] for c in sample]
    raw = generate(prompts, max_new=400)

    def parse_qa(s):
        m = re.search(r"\[.*\]", s, re.S)
        try:
            return [x for x in json.loads(m.group(0))
                    if isinstance(x.get("q"), str) and isinstance(x.get("a"), str)
                    and len(x["q"]) > 8 and len(x["a"]) > 8]
        except Exception:
            return []

    for c, s in zip(sample, raw):
        for qa in parse_qa(s):
            other = random.choice(chunks)
            ctx = [c, other] if other is not c else [c]
            random.shuffle(ctx)
            examples.append((qa["q"], ctx, f"{qa['a'].strip()}\n\nSource: {c['src']}"))
            if random.random() < NEG_RATIO and len(chunks) > 2:
                wrong = random.sample([x for x in chunks if x is not c], 2)
                examples.append((qa["q"], wrong, NO_INFO))
    print("Training examples:", len(examples))

In [ ]:
# 6. Fine-tune with QLoRA (skipped automatically if there is too little data)
TUNED = False
if MODE == "rag_plus_ft" and len(examples) >= MIN_EXAMPLES:
    from trl import SFTTrainer, SFTConfig
    rows = [{"text": tokenizer.apply_chat_template(
                build_messages(q, ctx) + [{"role": "assistant", "content": a}], tokenize=False)}
            for q, ctx, a in examples]
    ds = Dataset.from_list(rows).shuffle(seed=42)

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=42)

    args = dict(per_device_train_batch_size=2, gradient_accumulation_steps=4,
                num_train_epochs=EPOCHS, learning_rate=2e-4, warmup_steps=5, weight_decay=0.01,
                fp16=True, bf16=False, optim="adamw_8bit", logging_steps=10, seed=42,
                output_dir="/kaggle/working/ckpt", report_to="none", save_strategy="no",
                dataset_text_field="text")
    try:
        cfg = SFTConfig(max_seq_length=MAX_SEQ, **args)
    except TypeError:                                   # newer TRL renamed this argument
        cfg = SFTConfig(max_length=MAX_SEQ, **args)
    try:
        trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=cfg)
    except TypeError:
        trainer = SFTTrainer(model=model, processing_class=tokenizer, train_dataset=ds, args=cfg)
    try:                                                # learn from answers, not from the prompt
        from unsloth.chat_templates import train_on_responses_only
        trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                          response_part="<|im_start|>assistant\n")
    except Exception as e:
        print("train_on_responses_only skipped:", type(e).__name__)
    trainer.train()
    TUNED = True
elif MODE == "rag_plus_ft":
    print(f"Only {len(examples)} examples (need {MIN_EXAMPLES}). Skipping fine-tune and using RAG only.")

FastLanguageModel.for_inference(model)
print(f"Done. Elapsed {(time.time() - T0) / 60:.1f} min. Fine-tuned: {TUNED}")

In [ ]:
# 7. Test on your own questions (base model vs tuned model, same retrieved context)
for q in TEST_QUESTIONS:
    msgs = build_messages(q, retrieve(q))
    print("\nQ:", q)
    print("TUNED:", generate([msgs])[0])
    if TUNED:
        with model.disable_adapter():
            print("BASE :", generate([msgs])[0])

In [ ]:
# 8. Save deliverables (download these from the Output tab before wiping)
OUT = f"/kaggle/working/deliver_{CLIENT}"
os.makedirs(OUT, exist_ok=True)
json.dump(chunks, open(f"{OUT}/chunks.json", "w", encoding="utf-8"), ensure_ascii=False)
np.save(f"{OUT}/vectors.npy", vecs)
if TUNED and SAVE_ADAPTER:
    model.save_pretrained(f"{OUT}/lora_adapter")
    tokenizer.save_pretrained(f"{OUT}/lora_adapter")
if TUNED and EXPORT_GGUF:
    model.save_pretrained_gguf(f"{OUT}/gguf", tokenizer, quantization_method="q4_k_m")
shutil.make_archive(f"/kaggle/working/{CLIENT}_package", "zip", OUT)
print("Saved:", f"/kaggle/working/{CLIENT}_package.zip")

In [ ]:
# 9. Public chat (password protected, closes by itself)
import gradio as gr
from transformers import TextIteratorStreamer
from threading import Thread

def chat(message, history):
    text = tokenizer.apply_chat_template(build_messages(message, retrieve(message)),
                                         tokenize=False, add_generation_prompt=True)
    enc = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    Thread(target=model.generate, kwargs=dict(
        **enc, streamer=streamer, max_new_tokens=350, do_sample=True,
        temperature=0.2, top_p=0.9, use_cache=True)).start()
    out = ""
    for piece in streamer:
        out += piece
        yield out

demo = gr.ChatInterface(
    chat, title=f"{ORG} assistant",
    description=f"Ask a question about {ORG}. This demo closes in {SESSION_MINUTES} minutes.",
    examples=TEST_QUESTIONS[:3])

USER, PASSWORD = "client", secrets.token_urlsafe(8)
demo.queue()
demo.launch(share=True, auth=(USER, PASSWORD), prevent_thread_lock=True, quiet=True)
print("\nLINK    :", getattr(demo, "share_url", "(see Gradio output above)"))
print("USER    :", USER)
print("PASSWORD:", PASSWORD)
print(f"Closing in {SESSION_MINUTES} minutes...")
time.sleep(SESSION_MINUTES * 60)
demo.close()
print("Chat closed.")

In [ ]:
# 10. Wipe everything (run after you have downloaded the package)
CONFIRM_WIPE = False        # set to True, then run this cell
if CONFIRM_WIPE:
    for name in ["model", "trainer", "emb", "index", "vecs", "bm25", "docs", "chunks", "examples"]:
        globals().pop(name, None)
    gc.collect(); torch.cuda.empty_cache()
    for p in glob.glob("/kaggle/working/*"):
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)
    print("Working folder wiped.")
    print("Now: delete the private Kaggle Dataset, then stop the session.")